# Get data from website

National Disease Surveillance including all infectious disease data of Thailand: http://doe.moph.go.th/surdata/index.php

## Get disease list from index website

National Disease Surveillance (Report 506)Enter the 506 program: http://doe.moph.go.th/surdata/index.php

In [6]:
import requests
from bs4 import BeautifulSoup

# Target URL
url = 'http://doe.moph.go.th/surdata/index.php'

# Send HTTP request to retrieve the webpage content
response = requests.get(url)
response.encoding = 'utf-8'

# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Initialize a list to store disease names and their corresponding links
disease_list = []

# Find all tags that are links with the class 'e1'
for link in soup.find_all('a', class_='e1'):
    # get the full URL by joining the base URL with the relative URL
    disease_name = link.text.strip()
    disease_url = link['href']
    full_url = requests.compat.urljoin(url, disease_url)
    # get the historical URL by replacing 'disease.php?' with 'disease.php?dcontent=old&'
    historical_url = full_url.replace('disease.php?', 'disease.php?dcontent=old&')

    # Append the disease name and corresponding URLs to the list
    disease_list.append((disease_name, full_url, historical_url))

# Print the first 5 items in the disease_list
for item in disease_list[:5]:
    print(item)


('DHF Total', 'http://doe.moph.go.th/surdata/disease.php?ds=262766', 'http://doe.moph.go.th/surdata/disease.php?dcontent=old&ds=262766')
('Measles Total', 'http://doe.moph.go.th/surdata/disease.php?ds=2122', 'http://doe.moph.go.th/surdata/disease.php?dcontent=old&ds=2122')
('Encephalitis Total', 'http://doe.moph.go.th/surdata/disease.php?ds=2829', 'http://doe.moph.go.th/surdata/disease.php?dcontent=old&ds=2829')
('TB Total', 'http://doe.moph.go.th/surdata/disease.php?ds=323334', 'http://doe.moph.go.th/surdata/disease.php?dcontent=old&ds=323334')
('STI Total', 'http://doe.moph.go.th/surdata/disease.php?ds=37383940417980', 'http://doe.moph.go.th/surdata/disease.php?dcontent=old&ds=37383940417980')


## Get download link from disease website

Each disease has its own website, which including multiple files for download.

In [20]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import time
import logging

# Set up logging to only log errors to a file
logging.basicConfig(filename='./error_log.log', level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')

def get_file_url(history_url, retries=3, delay=5):
    """
    Attempt to fetch file download URLs from a historical page URL with retries.
    """
    try:
        response = requests.get(history_url)
        response.raise_for_status()  # Check for HTTP request errors
    except requests.RequestException as e:
        logging.error(f"Error fetching the page: {e}")
        if retries > 0:
            time.sleep(delay)  # Wait before retrying
            return get_file_url(history_url, retries - 1, delay)
        else:
            return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    file_urls = []

    for link in soup.find_all('a', target="_blank"):
        file_url = link.get('href')
        if file_url and file_url.endswith('.rtf'):
            full_file_url = urljoin(history_url, file_url)
            file_name = file_url.split('/')[-1]
            file_urls.append((file_name, full_file_url))
    
    return file_urls

def download_file(file_url, file_name, file_path='../Data/GetData'):
    """
    Download a file with retries and error handling.
    """
    retries = 5
    delay = 5
    for attempt in range(retries):
        try:
            response = requests.get(file_url)
            response.raise_for_status()
            # Ensure the directory exists
            os.makedirs(file_path, exist_ok=True)
            with open(os.path.join(file_path, file_name), 'wb') as file:
                file.write(response.content)
                # Log when failed downloads succeed
                if attempt > 0:
                    logging.info(f"Downloaded {file_name} after {attempt} attempts.")
                break
        except requests.RequestException as e:
            logging.error(f"Failed to download {file_name}: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                logging.error(f"Failed to download {file_name} after {retries} attempts.")

def download_files(history_url):
    """
    Download all files from a given historical URL.
    """
    file_urls = get_file_url(history_url)
    for file_name, file_url in file_urls:
        download_file(file_url, file_name)

# Test the functions with the first historical URL in the list
history_url = disease_list[0][2]
download_files(history_url)

## Downloading file from website

using multiprocessing to download multiple files at the same time.

In [21]:
# Using multiprocessing to download files
from multiprocessing import Pool

# Main function to use multiprocessing for handling multiple diseases
def main(disease_list):
    with Pool(processes=4) as pool:  # Adjust number of processes according to your system
        results = pool.map(process_disease, disease_list)
        pool.close()
        pool.join()
    print(results)

if __name__ == '__main__':
    # Your initial scraping logic to populate disease_list
    url = 'http://doe.moph.go.th/surdata/index.php'
    soup = fetch_url(url)
    disease_list = []

    for link in soup.find_all('a', class_='e1'):
        disease_name = link.text.strip()
        disease_url = link['href']
        full_url = urljoin(url, disease_url)
        historical_url = full_url.replace('disease.php?', 'disease.php?dcontent=old&')
        disease_list.append((disease_name, full_url, historical_url))
